## VaR Modeling - Historical simulation VaR

In [6]:
import yfinance as yf 
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

In [170]:
def extract_data(underlyings_list,valuation_date,look_back_period):
    # Default looking back period: 2 years 
    end_date  =  (datetime.strptime(valuation_date, "%Y-%m-%d") + timedelta(days=0)).strftime("%Y-%m-%d")
    start_date = (datetime.strptime(end_date, "%Y-%m-%d") - timedelta(days=365*look_back_period)).strftime("%Y-%m-%d")
    extract_list = []
    for values in underlyings_list:
        data = yf.download(values, start=start_date, end=end_date, interval="1d")
        # data = yf.download(values, period = '2y', interval ='1d')
        data = data['Close'][[values]].reset_index()
        data.columns = ['Date',str(values) + "_close"]
        extract_list.append(data)
        # print(data) 
    combined_df = extract_list[0].copy()
    for df in extract_list[1:]:
        combined_df = pd.merge(combined_df, df, on='Date', how='inner')
    combined_df = combined_df.sort_values('Date').reset_index(drop=True)
    return combined_df

In [172]:
#### using the portfolio_share_list 
def mth1_calculate_HVaR(underlyings_list,portfolio_share_list,time_horizon,alpha,extract_df):
    extract_df = extract_df.copy()
    valuation_date_df = extract_df.tail(1).copy()
    valuation_date_df.rename(columns={col: f'mtm_{col}' for col in valuation_date_df.columns}, inplace=True)
    # mtm_price_list = valuation_date_df.columns[1:].values
    mtm_price_list = [col for col in valuation_date_df.columns if '_close' in col]
    # print(mtm_price_list)
    # print(valuation_date_df)
    # print(portfolio_share_list)
    valuation_date_df['mtm_portfolio_value'] = valuation_date_df[mtm_price_list].dot(portfolio_share_list)

    for values in underlyings_list:
        columns_name = values+'_'+str(time_horizon)+'D_changes'
        extract_df[columns_name] = extract_df[str(values) + "_close"].shift(-time_horizon)/ extract_df[str(values) + "_close"]
        # print(columns_name)

    extract_df_new = extract_df.assign(**valuation_date_df.squeeze())

    simulated_price_list = []
    for values in underlyings_list:
        columns_name = values+'_'+ "simulated_price"
        extract_df_new[columns_name] = extract_df_new[values+'_'+str(time_horizon)+'D_changes']*extract_df_new["mtm_"+values+'_close'] 
        simulated_price_list.append(columns_name)
    extract_df_new['simulated_portfolio_value'] = extract_df_new[simulated_price_list].dot(portfolio_share_list)
    extract_df_new['simulated_pnl'] = extract_df_new['simulated_portfolio_value'] - extract_df_new['mtm_portfolio_value']
    
    extract_df_new_final = extract_df_new[['simulated_pnl']]
    extract_df_new_final= extract_df_new_final.dropna(subset=['simulated_pnl'])
    extract_df_new_final = extract_df_new_final.sort_values(by='simulated_pnl', ascending=True)
    HVaR = extract_df_new_final['simulated_pnl'].quantile(alpha)
    return HVaR

In [174]:
#### using mtm_portfolio_value and portfolio_weighting
def mth2_calculate_HVaR(underlyings_list,mtm_portfolio_value,portfolio_weighting,time_horizon,confident_interval,extract_df):
    extract_df = extract_df.copy()   
    changes_cols = []
    for values in underlyings_list:
        columns_name = values+'_'+str(time_horizon)+'D_changes'
        extract_df[columns_name] = extract_df[str(values) + "_close"].shift(-time_horizon)/ extract_df[str(values) + "_close"]
        changes_cols.append(columns_name)
    extract_df ['mtm_portfolio_value'] = mtm_portfolio_value
    
    # Compute weighted sum of the changes (dot product) for each row
    weighted_changes = extract_df[changes_cols].mul(portfolio_weighting, axis=1).sum(axis=1)
    extract_df['simulated_portfolio_value'] = extract_df['mtm_portfolio_value'] * weighted_changes
    extract_df['simulated_pnl'] = extract_df['simulated_portfolio_value'] - extract_df['mtm_portfolio_value']


    extract_df_final = extract_df.copy()
    extract_df_final = extract_df_final.dropna()
    extract_df_final = extract_df_final[['simulated_pnl']]
    extract_df_final= extract_df_final.dropna(subset=['simulated_pnl'])
    extract_df_final = extract_df_final.sort_values(by='simulated_pnl', ascending=True)
    HVaR = extract_df_final['simulated_pnl'].quantile(1-confident_interval)
    
    return  HVaR

In [192]:
underlyings_list = ["COIN","NFLX","RACE"]
portfolio_share_list = [24,140,12]
mtm_portfolio_value = 23006.240265
portfolio_weighting = [24*245.929993/mtm_portfolio_value,140*90.730003/mtm_portfolio_value,12*366.809998/mtm_portfolio_value]
confident_interval = 0.95
# valuation_date = "2025-12-31" 
valuation_date = str(datetime.today().strftime("%Y-%m-%d"))
look_back_period = 1
extract_df = extract_data(underlyings_list,valuation_date,look_back_period)
M1_HVaR_30D_95 = mth1_calculate_HVaR(underlyings_list, portfolio_share_list, 30, 1-confident_interval , extract_df)
M1_HVaR_10D_95 = mth1_calculate_HVaR(underlyings_list, portfolio_share_list, 10, 1-confident_interval  , extract_df)

M2_HVaR_30D_95 = mth2_calculate_HVaR(underlyings_list, mtm_portfolio_value, portfolio_weighting, 30, confident_interval , extract_df)
M2_HVaR_10D_95 = mth2_calculate_HVaR(underlyings_list, mtm_portfolio_value, portfolio_weighting, 10, confident_interval  , extract_df)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [197]:
print(f"Run Date: {str(datetime.today())}")
print(f"Look back period: {len(extract_df)} Day from {extract_df['Date'][0]} to {extract_df['Date'].iloc[-1]} ")
print("Method 1: Input Portfolio Share variable") 
print(f"risk indictor - 95% HVaR(30D) is :{M1_HVaR_30D_95}")
print(f"risk indictor - 95% HVaR(10D) is :{M1_HVaR_10D_95}")
print("Method 2: Input Portfolio Weighting varible and Portfolio Market Value variable") 
print(f"risk indictor - 95% HVaR(30D) is :{M2_HVaR_30D_95}")
print(f"risk indictor - 95% HVaR(10D) is :{M2_HVaR_10D_95}")

Run Date: 2026-01-08 17:17:53.810486
Look back period: 250 Day from 2025-01-08 00:00:00 to 2026-01-07 00:00:00 
Method 1: Input Portfolio Share variable
risk indictor - 95% HVaR(30D) is :-3964.8536830492744
risk indictor - 95% HVaR(10D) is :-2392.3345352805363
Method 2: Input Portfolio Weighting varible and Portfolio Market Value variable
risk indictor - 95% HVaR(30D) is :-3964.853713713516
risk indictor - 95% HVaR(10D) is :-2392.334570026728
